# Verdius v1 Pipeline — Article-Level Causal Extraction

End-to-end inference pipeline running entirely in Colab.

**Pipeline stages:**
1. Sentence split — spaCy sentencizer with char offsets
2. st0 filter — RoBERTa-large binary causal classifier (drops ~90%)
3. st1 + st2 — relation type classifier + BIO subject/object span extractor
4. Coref — fastcoref / LingMess over the full article
5. Resolve + dedup — match st2 spans to coref clusters via char-offset IoU; head-noun fallback
6. Build graph JSON — assemble (Article, Events, Causal Edges) per article
7. Write to Neo4j — `:Article -[:CONTAINS]-> :Event -[:CAUSES|ENABLES|PREVENTS|INTENDS]-> :Event`

**Schema:** simplest possible — no Entity nodes in v1. Each causal sentence emits two Event nodes (SUBJ and OBJ) and one typed causal edge. Coref still runs, but only to MERGE event nodes that refer to the same happening across sentences.

**Models:** loaded from Hugging Face Hub — `eitang/st0`, `eitang/st1`, `eitang/st2`. First call caches to the Colab runtime disk.

**Runtime:** any GPU works. L4 or A100 recommended (Colab Pro).

## 0. Confirm GPU

In [ ]:
import torch

assert torch.cuda.is_available(), 'No GPU detected — switch runtime to GPU.'
print('Device:', torch.cuda.get_device_name(0))
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## 1. Install dependencies

`torch` ships with Colab. We add spacy (sentence split), transformers (st0/st1/st2), fastcoref (coref), neo4j (graph sink).

In [ ]:
!pip install -q spacy==3.8.14 transformers==4.44.2 fastcoref neo4j

## 2. Clone the repo

Public — no auth needed. Pulls latest if already cloned.

In [ ]:
import os

if not os.path.isdir('Relation_extraction'):
    !git clone https://github.com/eitang5/Relation_extraction.git
else:
    print('Repo already cloned. Pulling latest...')
    !git -C Relation_extraction pull

%cd Relation_extraction

## 3. Mount Google Drive (for output)

Models load from HF Hub (no Drive needed for them). We still mount Drive so the final article JSON outputs persist across runtime restarts.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

out_root = '/content/drive/MyDrive/verdius/v1_outputs'
os.makedirs(out_root, exist_ok=True)
print(f'Output dir ready: {out_root}')

## 4. Sample article

Short news-style article (Iran / Hormuz / oil) with several causal claims. Running example through every cell below.

In [ ]:
SAMPLE_ARTICLE = (
    "The Strait of Hormuz remained closed for a third consecutive day on Wednesday "
    "after Iranian naval forces detained a Liberian-flagged tanker on Monday morning, "
    "sending Brent crude to a 14-month high of $112 a barrel. Saudi Aramco said it "
    "would temporarily reroute roughly 2 million barrels per day through the "
    "East-West Pipeline because of the closure. The detention followed a strike by "
    "Israeli aircraft on a missile depot in western Iran on Sunday night, which "
    "Tehran blamed for triggering its retaliation. European equity markets fell "
    "sharply, with the DAX closing 2.4% lower as investors priced in higher input costs."
)

ARTICLE_ID = 'demo_iran_hormuz'
ARTICLE_METADATA = {
    'title': 'Strait of Hormuz closure sends oil prices higher',
    'url': 'https://example.com/demo',
    'publication_date': '2026-05-06',
}
print(f'Article length: {len(SAMPLE_ARTICLE)} chars')

## 5. Module 1 — sentence split

spaCy blank('en') + sentencizer. Char offsets are load-bearing for downstream span resolution.

In [ ]:
from v1_pipeline.sentence_split import split_sentences

sentences = split_sentences(SAMPLE_ARTICLE)
for i, s in enumerate(sentences):
    s['idx'] = i  # attach idx so we can trace through later stages
    print(f'  [{i}] [{s["start"]:>3}:{s["end"]:>3}] {s["text"][:80]!r}{"..." if len(s["text"]) > 80 else ""}')
print(f'\nTotal: {len(sentences)} sentences')

## 6. Module 2 — st0 filter

Binary causal classifier (RoBERTa-large). Drops sentences predicted as `no_relation`. Expected ~90% drop rate on real articles; demo article will keep most because it's causal-dense.

In [ ]:
from v1_pipeline.st0_filter import filter_sentences

survivors = filter_sentences(sentences)
print(f'{len(survivors)} / {len(sentences)} sentences survived st0')
for s in survivors:
    print(f'  [{s["idx"]}] {s["text"][:90]!r}{"..." if len(s["text"]) > 90 else ""}')

## 7. Module 3 — st1 (relation type) + st2 (BIO subject/object spans)

**st1** predicts one of `cause / enable / prevent / intend / no_relation`. The `no_relation` ones get dropped (st0 and st1 occasionally disagree).

**st2** walks BIO tags to recover SUBJ and OBJ spans inside each sentence.

In [ ]:
from v1_pipeline.st1_classify import classify_relations
from v1_pipeline.st2_extract import extract_spans_batch

rel_results = classify_relations(survivors)
kept_sentences = [r['sentence'] for r in rel_results]
span_results = extract_spans_batch(kept_sentences)

print(f'After st1: {len(rel_results)} sentences kept')
for r, s in zip(rel_results, span_results):
    subj = s['subj_spans'][0][2] if s['subj_spans'] else '(none)'
    obj  = s['obj_spans'][0][2]  if s['obj_spans']  else '(none)'
    print(f'  [{r["sentence"]["idx"]}] {r["relation_type"]:8} (p={r["confidence"]:.2f})  {subj!r} → {obj!r}')

## 8. Module 4 — coref

fastcoref / LingMess clusters NP-shaped mentions across the whole article. We use clusters only to MERGE event nodes (two spans in the same cluster → one Event node). No Entity layer in v1.

First call downloads a longformer-based coref model (~500MB), cached for the session.

In [ ]:
from v1_pipeline.coref import coref_article

clusters = coref_article(SAMPLE_ARTICLE)
print(f'{len(clusters)} coref clusters\n')
for ci, cluster in enumerate(clusters):
    mentions = [SAMPLE_ARTICLE[s:e] for s, e in cluster]
    print(f'  cluster {ci}: {mentions}')

## 9. Modules 5 + 6 — resolver + dedup

**Resolver** matches each st2 SUBJ/OBJ span (article-global char offsets) to the best-overlapping coref cluster (IoU >= 0.5).

**Dedup** falls back to head-noun string match when a span doesn't fall into any coref cluster.

In [ ]:
from v1_pipeline.resolver import find_cluster_for_span
from v1_pipeline.dedup import head_noun

for r, s in zip(rel_results, span_results):
    sent_start = r['sentence']['start']
    if not s['subj_spans'] or not s['obj_spans']:
        continue
    subj = s['subj_spans'][0]
    obj  = s['obj_spans'][0]
    subj_g = (sent_start + subj[0], sent_start + subj[1])
    obj_g  = (sent_start + obj[0],  sent_start + obj[1])
    s_cluster = find_cluster_for_span(subj_g, clusters)
    o_cluster = find_cluster_for_span(obj_g,  clusters)
    s_key = f'cluster:{s_cluster}' if s_cluster is not None else f'noun:{head_noun(subj[2])}'
    o_key = f'cluster:{o_cluster}' if o_cluster is not None else f'noun:{head_noun(obj[2])}'
    print(f'  [{r["sentence"]["idx"]}] {subj[2]!r:35} → {s_key}')
    print(f'        {obj[2]!r:35} → {o_key}')

## 10. Module 7a — build article graph

Orchestrator: runs modules 1-6 end-to-end and emits the canonical article-graph dict (sentences + events + edges + stats). This is what gets written to disk as JSON and loaded into Neo4j.

In [ ]:
import json
from v1_pipeline.graph_build import build_article_graph

graph = build_article_graph(ARTICLE_ID, SAMPLE_ARTICLE, ARTICLE_METADATA)

print('Stats:', json.dumps(graph['stats'], indent=2))
print(f'\n{len(graph["events"])} events:')
for ev in graph['events']:
    print(f'  {ev["id"][-12:]}  {ev["name"]!r}  (in sentences {ev["sentence_indices"]})')
print(f'\n{len(graph["edges"])} causal edges:')
for e in graph['edges']:
    print(f'  {e["source"][-12:]} -[{e["relation_type"]}]-> {e["target"][-12:]}  (p={e["confidence"]:.2f})')

## 11. Save JSON and write to Neo4j

Two-step sink: persist the article graph as JSON to Drive (canonical audit log), then write the same data to Neo4j via `MERGE` (idempotent — safe to re-run).

**Set your dev Neo4j credentials below before running this cell.**

In [ ]:
import json
from pathlib import Path

# 1. Save JSON to Drive.
out_dir = Path('/content/drive/MyDrive/verdius/v1_outputs')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f'{ARTICLE_ID}.json'
out_path.write_text(json.dumps(graph, indent=2))
print(f'JSON written to {out_path} ({out_path.stat().st_size} bytes)')

# 2. Set Neo4j env vars (dev instance).
os.environ['NEO4J_URI'] = ''       # bolt://...
os.environ['NEO4J_USERNAME'] = 'neo4j'
os.environ['NEO4J_PASSWORD'] = ''  # paste from your dev creds
# os.environ['NEO4J_DATABASE'] = 'neo4j'  # uncomment if not default

if os.environ.get('NEO4J_URI') and os.environ.get('NEO4J_PASSWORD'):
    from v1_pipeline.neo4j_writer import write_article_to_neo4j
    counts = write_article_to_neo4j(graph)
    print(f'Neo4j writes: {counts}')
else:
    print('Skipping Neo4j write — NEO4J_URI / NEO4J_PASSWORD not set.')

## 12. Module 8 — multi-article runner

CLI form for batch jobs. Reads a JSONL where each line is `{"id": "...", "text": "...", ...metadata}`, writes one JSON per article to `--output-dir`, optionally to Neo4j.

Demo: write the sample article as a 1-line JSONL and process it.

In [ ]:
import json
from pathlib import Path

demo_jsonl = Path('/content/demo_articles.jsonl')
demo_jsonl.write_text(json.dumps({
    'id': ARTICLE_ID,
    'text': SAMPLE_ARTICLE,
    **ARTICLE_METADATA,
}) + '\n')

# Without --neo4j (JSON only). Add --neo4j once Neo4j env vars are set.
!python -m v1_pipeline.runner --input /content/demo_articles.jsonl --output-dir /content/drive/MyDrive/verdius/v1_outputs

## Next steps

**v1 done if every cell above ran cleanly.** Output you should see:
- Sample article splits into ~5-7 sentences
- st0 keeps most of them (article is causal-dense)
- st1 emits `cause`/`enable`/`prevent`/`intend` with confidences
- st2 picks SUBJ/OBJ spans per causal sentence
- coref finds clusters for Iran/Hormuz/etc
- graph_build returns ~5-10 events and ~3-5 causal edges
- JSON saved to Drive; Neo4j (if creds set) shows MERGE'd nodes

**Deferred to v2:**
- NER → real `:Entity` nodes (PERSON / ORG / GPE) with `PARTICIPATES_IN` edges
- Cross-article entity grounding (same `:Entity` across articles)
- Event coref (instead of head-noun string match)
- Time/modality/magnitude fields (currently LLM-only via v0)
- Multi-SUBJ / multi-OBJ within one sentence (currently take first only)

**Pick up code changes from local edits:**
```python
!git -C /content/Relation_extraction pull
# Force re-import of all v1 modules:
import importlib, sys
for mod in list(sys.modules):
    if mod.startswith('v1_pipeline'):
        importlib.reload(sys.modules[mod])
```